# 🍃 Taller NoSQL — Análisis con SQLite
**Materia:** Arquitectura de Datos | **Autor:** Daniel Alejandro Garcia | **ID:** 863128

Este notebook consulta `data/almacen.sqlite` generado por el ETL y produce reportes visuales.

> **Prerequisito:** ejecutar `python scripts/02_etl_mongo_sqlite.py` antes de correr este notebook.

In [ ]:
%matplotlib inline
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import os

# Conexion a SQLite — validar que el archivo exista
SQLITE_PATH = 'data/almacen.sqlite'
if not os.path.exists(SQLITE_PATH):
    raise FileNotFoundError(f'No encontrado: {SQLITE_PATH}. Ejecuta primero el ETL.')

conn = sqlite3.connect(SQLITE_PATH)
conn.execute('PRAGMA foreign_keys = ON')
print('Conexion a SQLite OK:', SQLITE_PATH)

## 1. Vista general de las tablas

In [ ]:
for tabla in ['personas', 'articulos', 'ventas']:
    df = pd.read_sql_query(f'SELECT * FROM {tabla}', conn)
    print(f'\n--- {tabla} ({len(df)} registros) ---')
    display(df)

## 2. Top 5 artículos más vendidos

In [ ]:
try:
    df_top5 = pd.read_sql_query("""
        SELECT a.nombreArticulo,
               SUM(v.cantidadProductos) AS total_vendido
        FROM ventas v
        JOIN articulos a ON v.idArticulo = a.idArticulo
        GROUP BY v.idArticulo, a.nombreArticulo
        ORDER BY total_vendido DESC
        LIMIT 5
    """, conn)

    plt.figure(figsize=(10, 5))
    plt.bar(df_top5['nombreArticulo'], df_top5['total_vendido'], color='#00ED64')
    plt.title('Top 5 Artículos Más Vendidos', fontsize=14, fontweight='bold')
    plt.xlabel('Artículo')
    plt.ylabel('Cantidad Vendida')
    plt.xticks(rotation=30, ha='right')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    display(df_top5)
except Exception as e:
    print(f'Error: {e}')
finally:
    pass

## 3. Top 5 artículos MENOS vendidos (LEFT JOIN + COALESCE)

In [ ]:
try:
    df_menos = pd.read_sql_query("""
        SELECT a.nombreArticulo,
               COALESCE(SUM(v.cantidadProductos), 0) AS total_vendido
        FROM articulos a
        LEFT JOIN ventas v ON a.idArticulo = v.idArticulo
        GROUP BY a.idArticulo
        ORDER BY total_vendido ASC
        LIMIT 5
    """, conn)

    plt.figure(figsize=(10, 5))
    plt.barh(df_menos['nombreArticulo'], df_menos['total_vendido'], color='#F85149')
    plt.title('Top 5 Artículos Menos Vendidos', fontsize=14, fontweight='bold')
    plt.xlabel('Cantidad Vendida')
    plt.ylabel('Artículo')
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    display(df_menos)
except Exception as e:
    print(f'Error: {e}')

## 4. Top 5 compradores con más compras

In [ ]:
try:
    df_comp = pd.read_sql_query("""
        SELECT p.nombre,
               COUNT(v.idVenta)         AS num_compras,
               SUM(v.cantidadProductos) AS total_productos
        FROM personas p
        JOIN ventas v ON p.numeroDocumento = v.idComprador
        GROUP BY p.numeroDocumento, p.nombre
        ORDER BY num_compras DESC
        LIMIT 5
    """, conn)

    plt.figure(figsize=(10, 5))
    plt.bar(df_comp['nombre'], df_comp['num_compras'], color='#58A6FF')
    plt.title('Top 5 Compradores con Más Compras', fontsize=14, fontweight='bold')
    plt.xlabel('Comprador')
    plt.ylabel('Número de Compras')
    plt.xticks(rotation=20, ha='right')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    display(df_comp)
except Exception as e:
    print(f'Error: {e}')

## 5. Histograma de distribución de ventas

In [ ]:
try:
    df_hist = pd.read_sql_query(
        'SELECT cantidadProductos FROM ventas', conn)

    plt.figure(figsize=(8, 5))
    plt.hist(df_hist['cantidadProductos'], bins=10, color='#F0B429', edgecolor='black')
    plt.title('Distribución de Ventas por Cantidad', fontsize=14, fontweight='bold')
    plt.xlabel('Cantidad de Productos')
    plt.ylabel('Frecuencia')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Error: {e}')

In [ ]:
# Cerrar conexion
conn.close()
print('Conexion cerrada.')